**1. Setup**

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report

**2. Load Data**

In [3]:
df = pd.read_csv("diabetes.csv")

In [4]:
df.shape

(768, 9)

In [5]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [6]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


**⚕️ Pima Indians Diabetes Dataset**

| Column Name | Description |
|------------|-------------|
| Pregnancies | Number of times the patient has been pregnant |
| Glucose | Plasma glucose concentration after 2 hours in an oral glucose tolerance test |
| BloodPressure | Diastolic blood pressure (mm Hg) |
| SkinThickness | Thickness of the triceps skin fold (mm) |
| Insulin | 2 hour serum insulin level (mu U/ml) |
| BMI | Body Mass Index (weight in kg / height in m²) |
| DiabetesPedigreeFunction | Likelihood of diabetes based on family history |
| Age | Age of the patient in years |
| Outcome | Target variable indicating diabetes status (1 = Diabetic, 0 = Non Diabetic) |

**3. Quick checks (high-level)**

In [7]:
# missing values
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [8]:
df["Outcome"].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [9]:
df["Outcome"].value_counts(normalize=True)*100

Outcome
0    65.104167
1    34.895833
Name: proportion, dtype: float64

**4. Split features and target**

In [10]:
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

In [11]:
X[:5]

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33


In [12]:
y[:5]

0    1
1    0
2    1
3    0
4    1
Name: Outcome, dtype: int64

**Model Selection Flow:**

Train-test split → model selection with Stratified K-Fold → hyperparameter tuning on the selected model → final evaluation on the untouched test set.

**5. Train Test Split**

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [14]:
print("Dataset size:", X.shape)
print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Dataset size: (768, 8)
Training set size: (614, 8)
Test set size: (154, 8)


**6. Stratifed K-Fold**

In [15]:
k = 5
cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

**7. Model Selection**

In [16]:
# models to try
models = {
    "LogisticRegression": LogisticRegression(),
    "SVC": SVC(),
    "RandomForestClassifier": RandomForestClassifier()
}

cv_scores = {}

# model selection process
for name, model in models.items():
    # print(name, model)
    # print("-"*40)

    # pipeline - scaler + model
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    # cross validation scores
    scores = cross_val_score(
        estimator=pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring="accuracy"
    )

    # print(f"{name} - cross val scores: {scores}")
    # print("-"*40)

    # mean_cross_val score for each model is saved in cv_scores
    cv_scores[name] = scores.mean()

    print("-"*40)
    print(name)
    print(f"Mean CV Accuracy: {scores.mean():.3f}")


# print(cv_scores)

----------------------------------------
LogisticRegression
Mean CV Accuracy: 0.788
----------------------------------------
SVC
Mean CV Accuracy: 0.778
----------------------------------------
RandomForestClassifier
Mean CV Accuracy: 0.757


In [17]:
cv_scores

{'LogisticRegression': 0.7882447021191524,
 'SVC': 0.7784886045581766,
 'RandomForestClassifier': 0.7573370651739304}

In [18]:
best_model_name = max(cv_scores, key=cv_scores.get)
print(f"Selected model: {best_model_name}")

Selected model: LogisticRegression


**8. Hyperparameter Tuning (only on best model)**

In [19]:
param_grid = {
    "LogisticRegression": {
        "model__C": [0.01, 0.1, 1, 10]
    },
    "SVC": {
        "model__C": [0.1, 1, 10],
        "model__kernel": ["linear", "rbf"]
    },
    "RandomForestClassifier": {
        "model__n_estimators": [100, 300, 500],
        "model__max_depth": [None, 5, 10],
        "model__min_samples_split": [2, 5, 10]
    }
}

In [20]:
models[best_model_name]

LogisticRegression()

In [21]:
param_grid[best_model_name]

{'model__C': [0.01, 0.1, 1, 10]}

In [22]:
# create a pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", models[best_model_name])
])

In [23]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid[best_model_name],
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

In [24]:
grid_search.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model', LogisticRegression())]),
             n_jobs=-1, param_grid={'model__C': [0.01, 0.1, 1, 10]},
             scoring='accuracy')

In [25]:
best_model = grid_search.best_estimator_

In [26]:
print("Hyper Parameter Tuning - results:")
print("Best CV Accuracy:", grid_search.best_score_)
print("Best Parameters:", grid_search.best_params_)

Hyper Parameter Tuning - results:
Best CV Accuracy: 0.7882447021191524
Best Parameters: {'model__C': 1}


**9. Final evaluation on undeen test data**

In [27]:
y_pred = best_model.predict(X_test)

print("FINAL TEST SET EVALUATION:")
print("Test accuracy:", accuracy_score(y_test, y_pred))
print("CLASSIFICATION REPORT")
print(classification_report(y_test, y_pred))

FINAL TEST SET EVALUATION:
Test accuracy: 0.7142857142857143
CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.76      0.82      0.79       100
           1       0.61      0.52      0.56        54

    accuracy                           0.71       154
   macro avg       0.68      0.67      0.67       154
weighted avg       0.71      0.71      0.71       154

